In [ ]:
########## __setup.py ##########

In [1]:
# Imports
from datetime import datetime, timedelta
import geopandas as gpd
import glob
import numpy as np
import pandas as pd
import shapely

# Set the data folder
data_folder = '~/DansData/'
data_folder

'~/DansData/'

In [ ]:
########## a_downsizer.py ##########

In [5]:
data = pd.read_csv(data_folder + 'BOF/Dan & Kelsey All FUNDY data 05-19-2023.CSV',
                usecols=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9,  # only load the necessary columns
                            10, 11, 12, 13, 14, 15, 18, 19,
                            20, 21, 22,
                            42, 43, 44, 45],
                   na_values=('.', ' .', '  .', '   .', '    .', '     .'),
                   # header=None,
                   # on_bad_lines='skip',
                   engine='python'
                   )
# Remove dud last row
print(data.iloc[-1:])
data = data.iloc[:-1]
data

       FILEID  EVENTNO  MONTH  DAY  YEAR  GMT  LATITUDE  LONGITUDE  LEGTYPE  \
481781            NaN    NaN  NaN   NaN  NaN       NaN        NaN      NaN   

        LEGSTAGE  ...  BEAUFORT  SIGHTNO SPECCODE  IDREL  NUMBER  CONFIDNC  \
481781       NaN  ...       NaN      NaN      NaN    NaN     NaN       NaN   

        TYPE PLATFORM  DDSOURCE  IDSOURCE  
481781   NaN      NaN       NaN       NaN  

[1 rows x 25 columns]


,FILEID,EVENTNO,MONTH,DAY,YEAR,GMT,LATITUDE,LONGITUDE,LEGTYPE,LEGSTAGE,...,BEAUFORT,SIGHTNO,SPECCODE,IDREL,NUMBER,CONFIDNC,TYPE,PLATFORM,DDSOURCE,IDSOURCE
0,A179034,180.0,2.0,3.0,1979.0,154000.0,44.16667,-67.15000,2.0,2.0,...,6.0,NaN,NaN,NaN,NaN,NaN,aerial,627.0,CET,CET
1,A179034,190.0,2.0,3.0,1979.0,154500.0,44.01667,-66.95000,2.0,2.0,...,6.0,NaN,NaN,NaN,NaN,NaN,aerial,627.0,CET,CET
2,A179034,280.0,2.0,3.0,1979.0,163000.0,44.08333,-67.10000,2.0,2.0,...,5.0,NaN,NaN,NaN,NaN,NaN,aerial,627.0,CET,CET
3,A179034,290.0,2.0,3.0,1979.0,163500.0,44.18333,-67.25000,2.0,2.0,...,5.0,NaN,NaN,NaN,NaN,NaN,aerial,627.0,CET,CET
4,A179034,300.0,2.0,3.0,1979.0,164000.0,44.28333,-67.38333,2.0,2.0,...,5.0,NaN,NaN,NaN,NaN,NaN,aerial,627.0,CET,CET
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
481776,o9003030,597.0,10.0,3.0,2000.0,232100.0,44.60000,-66.59900,NaN,NaN,...,NaN,597.0,RIWH,3.0,1.0,0.0,tagtrk,802.0,NEA,TAG
481777,o9023120,598.0,8.0,21.0,2002.0,234600.0,44.53700,-66.41600,NaN,NaN,...,NaN,598.0,RIWH,3.0,1.0,0.0,tagtrk,802.0,NEA,TAG
481778,o9023120,599.0,8.0,22.0,2002.0,172600.0,44.62700,-66.47000,NaN,NaN,...,NaN,599.0,RIWH,3.0,1.0,0.0,tagtrk,802.0,NEA,TAG
481779,o9023120,600.0,8.0,23.0,2002.0,82500.0,44.64300,-66.42800,NaN,NaN,...,NaN,600.0,RIWH,3.0,1.0,0.0,tagtrk,802.0,NEA,TAG


In [6]:
# Downcast float and integer columns to most efficient data type
float_cols = data.select_dtypes('float').columns
integer_cols = data.select_dtypes('integer').columns
data.loc[:, float_cols] = data[float_cols].apply(pd.to_numeric, downcast='float')
data.loc[:, integer_cols] = data[integer_cols].apply(pd.to_numeric, downcast='integer')


In [8]:
#####
# Rename, reformat, rearrange

# Rename columns
data.columns = ['file_id', 'event_no', 'month', 'day', 'year', 'time', 'lat', 'lon',
                'leg_type', 'leg_stage',
                'alt', 'heading', 'wx', 'cloud', 'vis', 'bss',
                'si_no', 'sp_code', 'sp_rel', 'no', 'no_conf',
                # 'beh01', 'beh02', 'beh03', 'beh04', 'beh05',
                # 'beh06', 'beh07', 'beh08', 'beh09', 'beh10',
                # 'beh11', 'beh12', 'beh13', 'beh14', 'beh15',
                'survey_type', 'platform', 'dd_source', 'id_source']

In [9]:
data

,file_id,event_no,month,day,year,time,lat,lon,leg_type,leg_stage,...,bss,si_no,sp_code,sp_rel,no,no_conf,survey_type,platform,dd_source,id_source
0,A179034,180.0,2.0,3.0,1979.0,154000.0,44.166672,-67.150002,2.0,2.0,...,6.0,NaN,NaN,NaN,NaN,NaN,aerial,627.0,CET,CET
1,A179034,190.0,2.0,3.0,1979.0,154500.0,44.016670,-66.949997,2.0,2.0,...,6.0,NaN,NaN,NaN,NaN,NaN,aerial,627.0,CET,CET
2,A179034,280.0,2.0,3.0,1979.0,163000.0,44.083328,-67.099998,2.0,2.0,...,5.0,NaN,NaN,NaN,NaN,NaN,aerial,627.0,CET,CET
3,A179034,290.0,2.0,3.0,1979.0,163500.0,44.183331,-67.250000,2.0,2.0,...,5.0,NaN,NaN,NaN,NaN,NaN,aerial,627.0,CET,CET
4,A179034,300.0,2.0,3.0,1979.0,164000.0,44.283329,-67.383331,2.0,2.0,...,5.0,NaN,NaN,NaN,NaN,NaN,aerial,627.0,CET,CET
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
481776,o9003030,597.0,10.0,3.0,2000.0,232100.0,44.599998,-66.598999,NaN,NaN,...,NaN,597.0,RIWH,3.0,1.0,0.0,tagtrk,802.0,NEA,TAG
481777,o9023120,598.0,8.0,21.0,2002.0,234600.0,44.536999,-66.416000,NaN,NaN,...,NaN,598.0,RIWH,3.0,1.0,0.0,tagtrk,802.0,NEA,TAG
481778,o9023120,599.0,8.0,22.0,2002.0,172600.0,44.626999,-66.470001,NaN,NaN,...,NaN,599.0,RIWH,3.0,1.0,0.0,tagtrk,802.0,NEA,TAG
481779,o9023120,600.0,8.0,23.0,2002.0,82500.0,44.643002,-66.428001,NaN,NaN,...,NaN,600.0,RIWH,3.0,1.0,0.0,tagtrk,802.0,NEA,TAG


In [ ]:
# # Concatenate behaviour columns into one
# data['beh'] = [[str(int(b)).zfill(2) for b in row if b == b]
#                for row in data[['beh01', 'beh02', 'beh03']].values.tolist()]
# data['beh'] = [';'.join(map(str, i)) for i in data['beh']]


In [10]:
# Rearrange columns
data = data[['file_id', 'survey_type', 'platform', 'dd_source', 'id_source',  # source
             'leg_type', 'leg_stage',  # survey
             'year', 'month', 'day', 'time',  # datetime
             'lat', 'lon',  # location
             'event_no', 'si_no',  # event identifiers
             'sp_code', 'sp_rel',  # sighting data (species)
             'no', 'no_conf',  # 'no_calves', 'beh',  # sighting data (number and behaviour)
             'heading', 'alt', 'wx', 'cloud', 'vis', 'bss',  # environmental data
             ]]

In [11]:
# Restrict to shipboard surveys (remove opportunistic data)
data = data[data['survey_type'] == 'shipbd']

# Restrict to Nereid
data = data[data['platform'] == 99]


In [16]:
#####
# Output
output_path = data_folder + 'BOF/BOF_sesienv/a_downsized/BOF_all.csv'
output_path
data.to_csv(output_path, index=False)

In [17]:
whos

Variable       Type         Data/Info
-------------------------------------
data           DataFrame            file_id survey_ty<...>383611 rows x 25 columns]
data_folder    str          ~/DansData/
datetime       type         <class 'datetime.datetime'>
float_cols     Index        Index(['EVENTNO', 'MONTH'<...>],\n      dtype='object')
glob           module       <module 'glob' from '/srv<...>/lib/python3.10/glob.py'>
gpd            module       <module 'geopandas' from <...>s/geopandas/__init__.py'>
integer_cols   Index        Index([], dtype='object')
np             module       <module 'numpy' from '/sr<...>kages/numpy/__init__.py'>
output_path    str          ~/DansData/BOF/BOF_sesien<...>v/a_downsized/BOF_all.csv
pd             module       <module 'pandas' from '/s<...>ages/pandas/__init__.py'>
shapely        module       <module 'shapely' from '/<...>ges/shapely/__init__.py'>
timedelta      type         <class 'datetime.timedelta'>


In [ ]:
########## b_onoff.py ##########

In [72]:
#####
# Input data
data = pd.read_csv(data_folder + 'BOF/BOF_sesienv/a_downsized/BOF_all.csv')
data

,file_id,survey_type,platform,dd_source,id_source,leg_type,leg_stage,year,month,day,...,sp_code,sp_rel,no,no_conf,heading,alt,wx,cloud,vis,bss
0,P187203,shipbd,99.0,NEA,RWC,5.0,1.0,1987.0,7.0,22.0,...,NaN,NaN,NaN,NaN,100.0,NaN,X,4.0,-1.0,2.0
1,P187203,shipbd,99.0,NEA,RWC,5.0,2.0,1987.0,7.0,22.0,...,NaN,NaN,NaN,NaN,100.0,NaN,X,4.0,-1.0,2.0
2,P187203,shipbd,99.0,NEA,RWC,5.0,2.0,1987.0,7.0,22.0,...,NaN,NaN,NaN,NaN,100.0,NaN,X,4.0,-1.0,1.0
3,P187203,shipbd,99.0,NEA,RWC,5.0,2.0,1987.0,7.0,22.0,...,NaN,NaN,NaN,NaN,140.0,NaN,X,4.0,-1.0,1.0
4,P187203,shipbd,99.0,NEA,RWC,5.0,2.0,1987.0,7.0,22.0,...,NaN,NaN,NaN,NaN,100.0,NaN,X,4.0,-1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
383606,p120256,shipbd,99.0,NEA,RWC,5.0,2.0,2020.0,9.0,12.0,...,NaN,NaN,NaN,NaN,282.0,NaN,C,2.0,5.0,2.0
383607,p120256,shipbd,99.0,NEA,RWC,5.0,2.0,2020.0,9.0,12.0,...,NaN,NaN,NaN,NaN,290.0,NaN,C,2.0,5.0,2.0
383608,p120256,shipbd,99.0,NEA,RWC,5.0,2.0,2020.0,9.0,12.0,...,NaN,NaN,NaN,NaN,291.0,NaN,C,2.0,5.0,2.0
383609,p120256,shipbd,99.0,NEA,RWC,5.0,2.0,2020.0,9.0,12.0,...,NaN,NaN,NaN,NaN,300.0,NaN,C,2.0,5.0,2.0


In [73]:
#####
# ON/OFF: determine if each point is ON or OFF
data.insert(7, 'on_off', 'OFF')  # create ON/OFF column with all entries set as OFF (as default)
data.loc[(
    (data['bss'] < 4)
    & (data['vis'] >= 2)
    & (data['leg_type'].isin([5, 6]) & ((data['leg_stage'] == 1) | (data['leg_stage'] == 2) | (data['leg_stage'] == 5)))
    & ((data['sp_rel'] == 3) | data['sp_rel'].isna())
), 'on_off'] = 'ON'


In [74]:
data.iloc[1:10,1:10]

,survey_type,platform,dd_source,id_source,leg_type,leg_stage,on_off,year,month
1,shipbd,99.0,NEA,RWC,5.0,2.0,OFF,1987.0,7.0
2,shipbd,99.0,NEA,RWC,5.0,2.0,OFF,1987.0,7.0
3,shipbd,99.0,NEA,RWC,5.0,2.0,OFF,1987.0,7.0
4,shipbd,99.0,NEA,RWC,5.0,2.0,OFF,1987.0,7.0
5,shipbd,99.0,NEA,RWC,5.0,2.0,OFF,1987.0,7.0
6,shipbd,99.0,NEA,RWC,5.0,2.0,OFF,1987.0,7.0
7,shipbd,99.0,NEA,RWC,5.0,2.0,OFF,1987.0,7.0
8,shipbd,99.0,NEA,RWC,5.0,2.0,OFF,1987.0,7.0
9,shipbd,99.0,NEA,RWC,5.0,2.0,OFF,1987.0,7.0


In [75]:
# Process ON/OFF
data['on_off_same'] = data['on_off'].eq(data['on_off'].shift())  # determine changes from ON to OFF or vv
data_OFF = data.copy()[data['on_off'] == 'OFF']  # create a dataframe of only OFF entries
data_OFF.drop(['on_off_same'], axis=1, inplace=True)
data_OFF.to_csv(data_folder + 'BOF/BOF_sesienv/b_onoff/BOF_OFF_se.csv', index=False)  # output OFF
data = data.copy()[data['on_off'] == 'ON'].reset_index(drop=True)  # keep only points that are ON


In [76]:
#####
# Datetimes

# Create datetimes
data.insert(8, 'datetime', '')
data['year'] = data['year'].astype(int).astype(str)  # reformat
data['month'] = data['month'].astype(int).astype(str).str.zfill(2)
data['day'] = data['day'].astype(int).astype(str).str.zfill(2)
data['time'] = data['time'].astype(int).astype(str).str.zfill(6)
data['datetime'] = (data['year'] + '-' + data['month'] + '-' + data['day'] + ' ' + data['time'])  # str in UTC
data['datetime'] = data['datetime'].apply(lambda d: datetime.strptime(d, '%Y-%m-%d %H%M%S'))  # datetime in UTC
data['datetime'] = data['datetime'].apply(lambda d: d - timedelta(hours=5))  # datetime in EST

# Create date, year, and month columns as strings in EST
data.drop(['year', 'month', 'day', 'time'], axis=1, inplace=True)  # remove columns in UTC
data.insert(9, 'date', '')
data.insert(10, 'year', '')
data.insert(11, 'month', '')
data.insert(12, 'day', '')
data.insert(13, 'time', '')
data['date'] = data['datetime'].apply(lambda d: d.strftime('%Y-%m-%d'))  # string in EST
data['year'] = data['datetime'].apply(lambda d: d.strftime('%Y'))
data['month'] = data['datetime'].apply(lambda d: d.strftime('%m'))
data['day'] = data['datetime'].apply(lambda d: d.strftime('%d'))
data['time'] = data['datetime'].apply(lambda d: d.strftime('%H:%M:%S'))


In [77]:
data

,file_id,survey_type,platform,dd_source,id_source,leg_type,leg_stage,on_off,datetime,date,...,sp_rel,no,no_conf,heading,alt,wx,cloud,vis,bss,on_off_same
0,p100205,shipbd,99.0,NEA,RWC,5.0,2.0,ON,2000-07-23 10:37:00,2000-07-23,...,3.0,2.0,0.0,180.0,NaN,X,9.0,2.0,3.0,False
1,p104217,shipbd,99.0,NEA,RWC,5.0,1.0,ON,2004-08-04 11:07:37,2004-08-04,...,NaN,NaN,NaN,91.0,NaN,H,2.0,5.0,1.0,False
2,p104217,shipbd,99.0,NEA,RWC,5.0,2.0,ON,2004-08-04 11:07:37,2004-08-04,...,NaN,NaN,NaN,91.0,NaN,H,2.0,5.0,1.0,True
3,p104217,shipbd,99.0,NEA,RWC,5.0,2.0,ON,2004-08-04 11:08:37,2004-08-04,...,NaN,NaN,NaN,94.0,NaN,H,2.0,5.0,1.0,True
4,p104217,shipbd,99.0,NEA,RWC,5.0,2.0,ON,2004-08-04 11:09:37,2004-08-04,...,NaN,NaN,NaN,90.0,NaN,H,2.0,5.0,1.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272783,p120256,shipbd,99.0,NEA,RWC,5.0,2.0,ON,2020-09-12 17:42:11,2020-09-12,...,NaN,NaN,NaN,282.0,NaN,C,2.0,5.0,2.0,True
272784,p120256,shipbd,99.0,NEA,RWC,5.0,2.0,ON,2020-09-12 17:42:41,2020-09-12,...,NaN,NaN,NaN,290.0,NaN,C,2.0,5.0,2.0,True
272785,p120256,shipbd,99.0,NEA,RWC,5.0,2.0,ON,2020-09-12 17:43:11,2020-09-12,...,NaN,NaN,NaN,291.0,NaN,C,2.0,5.0,2.0,True
272786,p120256,shipbd,99.0,NEA,RWC,5.0,2.0,ON,2020-09-12 17:43:41,2020-09-12,...,NaN,NaN,NaN,300.0,NaN,C,2.0,5.0,2.0,True


In [78]:
#####
# Seasons
data.insert(14, 'season', '')
data['season'] = data['month']  # <><>get Dan's season making function
data.insert(15, 'sn_id', '')
data['sn_id'] = data.apply(lambda r: 'sn' + str(r['season']).zfill(2), axis=1)


In [79]:
#####
# Surveys
sv_ids = data.copy().groupby(['year', 'sn_id']).agg({'file_id': 'unique'})
sv_ids['sv_id'] = sv_ids.apply(lambda r: ['sv' + str(i).zfill(2) for i in range(1, len(r['file_id']) + 1)], axis=1)
sv_ids = sv_ids.explode(['file_id', 'sv_id']).reset_index()
sv_ids = pd.merge(data[['sn_id', 'file_id']], sv_ids, on=['sn_id', 'file_id'])
data.insert(16, 'sv_id', sv_ids['sv_id'])


In [80]:
data

,file_id,survey_type,platform,dd_source,id_source,leg_type,leg_stage,on_off,datetime,date,...,sp_rel,no,no_conf,heading,alt,wx,cloud,vis,bss,on_off_same
0,p100205,shipbd,99.0,NEA,RWC,5.0,2.0,ON,2000-07-23 10:37:00,2000-07-23,...,3.0,2.0,0.0,180.0,NaN,X,9.0,2.0,3.0,False
1,p104217,shipbd,99.0,NEA,RWC,5.0,1.0,ON,2004-08-04 11:07:37,2004-08-04,...,NaN,NaN,NaN,91.0,NaN,H,2.0,5.0,1.0,False
2,p104217,shipbd,99.0,NEA,RWC,5.0,2.0,ON,2004-08-04 11:07:37,2004-08-04,...,NaN,NaN,NaN,91.0,NaN,H,2.0,5.0,1.0,True
3,p104217,shipbd,99.0,NEA,RWC,5.0,2.0,ON,2004-08-04 11:08:37,2004-08-04,...,NaN,NaN,NaN,94.0,NaN,H,2.0,5.0,1.0,True
4,p104217,shipbd,99.0,NEA,RWC,5.0,2.0,ON,2004-08-04 11:09:37,2004-08-04,...,NaN,NaN,NaN,90.0,NaN,H,2.0,5.0,1.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272783,p120256,shipbd,99.0,NEA,RWC,5.0,2.0,ON,2020-09-12 17:42:11,2020-09-12,...,NaN,NaN,NaN,282.0,NaN,C,2.0,5.0,2.0,True
272784,p120256,shipbd,99.0,NEA,RWC,5.0,2.0,ON,2020-09-12 17:42:41,2020-09-12,...,NaN,NaN,NaN,290.0,NaN,C,2.0,5.0,2.0,True
272785,p120256,shipbd,99.0,NEA,RWC,5.0,2.0,ON,2020-09-12 17:43:11,2020-09-12,...,NaN,NaN,NaN,291.0,NaN,C,2.0,5.0,2.0,True
272786,p120256,shipbd,99.0,NEA,RWC,5.0,2.0,ON,2020-09-12 17:43:41,2020-09-12,...,NaN,NaN,NaN,300.0,NaN,C,2.0,5.0,2.0,True


In [81]:
#####
# Sections

# Reformat for creating section ID
data['event_no'] = data['event_no'].astype(int).astype(str).str.zfill(4)

# Determine if each point is a change from...
data['date_same'] = data['date'].eq(data['date'].shift())  # ...one day to the next
data['leg_type_same'] = data['leg_type'].eq(data['leg_type'].shift())  # ...one leg type to another

# Create section IDs
data.loc[(  # entries where...
        ~data['date_same']  # ...the date changes...
        | ~data['leg_type_same']  # ...or the leg type changes...
        | data['leg_stage'].isin([1, 4])  # ...or a line begins or resumes...
        | ~data['on_off_same']  # ...or the survey changes from ON to OFF or vice versa...
),  # ...mark a new section and so...
    'section_id'] = data['file_id'] + '_' + data['date'] + '_' + data['event_no']  # ...get a section ID
data['section_id'] = data['section_id'].ffill()  # remaining entries get the section ID of the first entry
data.insert(5, 'section_id', data.pop('section_id'))  # relocate section ID column

# Remove same columns
data.drop(['date_same', 'leg_type_same', 'on_off_same'], axis=1, inplace=True)

# Reformat data types
data['leg_stage'] = data['leg_stage'].fillna(0)
for col in ['platform', 'leg_type', 'leg_stage', 'vis', 'bss']:
    data[col] = data[col].astype(int)
data.insert(6, 'leg_ts', '')
data['leg_ts'] = 'T' + data['leg_type'].astype(str) + 'S' + data['leg_stage'].astype(str)


#####
# Output
data.to_csv(data_folder + 'BOF/BOF_sesienv/b_onoff/BOF_ON_se.csv', index=False)


In [82]:
#####
# Sightings

# Keep only sightings
sightings = data.copy().dropna(subset='sp_code')

# Output
sightings.to_csv(data_folder + 'BOF/BOF_sesienv/b_onoff/BOF_ON_si.csv', index=False)
